# 18. 전체 게임 장르별 분포

**분석 목적:** 초기 반응 그룹과 침묵 그룹을 합산한 전체 인디게임을 대상으로 장르별 게임 수 분포를 확인한다.

**사용 데이터:**
- `data/preprocessed/steam_indie_games.csv` — 초기 반응 그룹 (리뷰 10개 이상)
- `data/preprocessed/steam_indie_games_silence.csv` — 침묵 그룹 (리뷰 10개 미만)

**분석 방법:** 다중 장르 중복 집계 — 게임이 여러 장르를 가지면 각 장르에 모두 포함

In [1]:
import ast
import warnings

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

EXCLUDE_GENRES = {'Indie'}

PALETTE = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B2', '#937860', '#DA8BC3', '#8C8C8C',
]

## 1. 데이터 로드 및 병합

In [2]:
df_response = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')
df_silence  = pd.read_csv('../../data/preprocessed/steam_indie_games_silence.csv')

df_response['response_group'] = '초기 반응 (≥10개)'
df_silence['response_group']  = '침묵 (<10개)'

df = pd.concat([df_response, df_silence], ignore_index=True)
df['genre_list'] = df['genres'].apply(lambda v: ast.literal_eval(v) if pd.notna(v) else [])

print(f'초기 반응 그룹 : {len(df_response):,}개')
print(f'침묵 그룹      : {len(df_silence):,}개')
print(f'전체           : {len(df):,}개')

초기 반응 그룹 : 8,730개
침묵 그룹      : 6,676개
전체           : 15,406개


## 2. 장르별 공급(게임 수) vs 수요(총 리뷰 수) — 이중 바 차트

In [3]:
df_exploded = (
    df.explode('genre_list')
    .rename(columns={'genre_list': 'genre'})
    .dropna(subset=['genre'])
)
df_exploded = df_exploded[~df_exploded['genre'].isin(EXCLUDE_GENRES)]

genre_stats = (
    df_exploded.groupby('genre')
    .agg(
        game_count=('appid', 'nunique'),
        total_reviews=('total_reviews', 'sum'),
    )
    .reset_index()
)

# 각각 내림차순 정렬
by_games   = genre_stats.sort_values('game_count', ascending=False).reset_index(drop=True)
by_reviews = genre_stats.sort_values('total_reviews', ascending=False).reset_index(drop=True)

color_map = {g: PALETTE[i] for i, g in enumerate(by_games['genre'])}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        '공급: 장르별 게임 수 (다중 집계)',
        '수요: 장르별 총 리뷰 수 (다중 집계)',
    ),
    horizontal_spacing=0.12,
)

for i, row in by_games.iterrows():
    fig.add_trace(go.Bar(
        x=[row['genre']],
        y=[row['game_count']],
        marker_color=color_map[row['genre']],
        text=[f"{row['game_count']:,}"],
        textposition='outside',
        showlegend=False,
        name=row['genre'],
    ), row=1, col=1)

for i, row in by_reviews.iterrows():
    fig.add_trace(go.Bar(
        x=[row['genre']],
        y=[row['total_reviews']],
        marker_color=color_map[row['genre']],
        text=[f"{row['total_reviews']:,}"],
        textposition='outside',
        showlegend=False,
        name=row['genre'],
    ), row=1, col=2)

fig.update_layout(
    title='장르별 시장 현황: 공급 vs 수요<br><sub>전체 15,406개 / 다중 장르 중복 집계 / Indie 태그 제외</sub>',
    height=520,
    bargap=0.3,
)
fig.update_yaxes(title_text='게임 수', row=1, col=1)
fig.update_yaxes(title_text='총 리뷰 수', row=1, col=2)
fig.show()

print('장르별 공급 vs 수요:')
summary = genre_stats.copy()
summary['공급 순위'] = summary['game_count'].rank(ascending=False).astype(int)
summary['수요 순위'] = summary['total_reviews'].rank(ascending=False).astype(int)
summary['순위 차이'] = summary['공급 순위'] - summary['수요 순위']
display(
    summary.sort_values('수요 순위')
    .rename(columns={'genre': '장르', 'game_count': '게임 수', 'total_reviews': '총 리뷰 수'})
    .set_index('장르')
)

장르별 공급 vs 수요:


,게임 수,총 리뷰 수,공급 순위,수요 순위,순위 차이
장르,,,,,
Adventure,7361,4125139,2,1,1
Action,6895,4032794,3,2,1
Simulation,3544,3013753,4,3,1
RPG,3101,2492253,6,4,2
Strategy,3327,2061354,5,5,0
Casual,7459,1797395,1,6,-5
Racing,557,171831,8,7,1
Sports,572,86395,7,8,-1


**이중 바 차트 해석:** 순위 차이(공급 순위 - 수요 순위)가 양수인 장르는 게임 수 대비 리뷰가 더 많이 몰리고, 음수인 장르는 게임은 많지만 리뷰가 상대적으로 적다.

**산점도 해석:** 점선(중앙값 기준선)을 기준으로 각 장르의 게임 수와 총 리뷰 수 위치를 확인할 수 있다. Casual은 게임 수가 가장 많지만 총 리뷰 수는 중앙값 아래에 위치하며, RPG·Simulation·Strategy는 게임 수가 적음에도 총 리뷰 수가 중앙값 이상이다.

## 3. 공급 vs 수요 포지셔닝 산점도

In [11]:
mid_games   = genre_stats['game_count'].median()
mid_reviews = genre_stats['total_reviews'].median()

genre_stats['reviews_per_game'] = (genre_stats['total_reviews'] / genre_stats['game_count']).round(0).astype(int)

color_map = {g: PALETTE[i] for i, g in enumerate(genre_stats.sort_values('game_count', ascending=False)['genre'])}

fig = go.Figure()

for _, row in genre_stats.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['game_count']],
        y=[row['total_reviews']],
        mode='markers+text',
        name=row['genre'],
        marker=dict(size=18, color=color_map[row['genre']], opacity=0.85, line=dict(width=1.5, color='white')),
        text=[row['genre']],
        textposition='top center',
        textfont=dict(size=12),
        customdata=[[row['reviews_per_game']]],
        hovertemplate=(
            '<b>%{text}</b><br>'
            '게임 수: %{x:,}개<br>'
            '총 리뷰 수: %{y:,}<br>'
            '게임당 평균 리뷰: %{customdata[0]:,}<extra></extra>'
        ),
        showlegend=False,
    ))

fig.add_vline(x=mid_games,   line_dash='dot', line_color='#adb5bd', line_width=1.5,
              annotation_text=f'게임 수 중앙값 ({int(mid_games):,})', annotation_position='top right',
              annotation_font=dict(size=10, color='#6B7280'))
fig.add_hline(y=mid_reviews, line_dash='dot', line_color='#adb5bd', line_width=1.5,
              annotation_text=f'총 리뷰 수 중앙값 ({int(mid_reviews):,})', annotation_position='top right',
              annotation_font=dict(size=10, color='#6B7280'))

fig.update_layout(
    # title='장르별 게임 수(공급) vs 리뷰 수(수요)<br><sub>점선: 각 축 중앙값</sub>',
    xaxis_title='게임 수 (공급)',
    yaxis_title='총 리뷰 수 (수요)',
    height=540,
    plot_bgcolor='#FAFAFA',
)
fig.show()